# <center> Simulation Video

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import networkx as nx
import matplotlib.animation as animation
from tqdm import tqdm
import pickle
import json

# ==============================================================================
# GLOBAL PARAMETERS
# ==============================================================================

# Scenario selection — change to switch between simulation results:
#   'scenario_baseline'
#   'scenario_suez_50pct_reduction'
#   'scenario_hormuz_closure_permanent'
SCENARIO = 'scenario_suez_evergiven'
SIMULATION_DIR = f'../part_4_new_simulation/simulation_output_data/{SCENARIO}'


# Video parameters
N_FRAMES = 480  # Number of frames to render (adjust this to control video length)
                 # Set to None to render all timesteps
FPS = 20         # Frames per second for the video

# Edge width metric choice
EDGE_METRIC = 'weight'  # Options: 'weight' (metric tons) or 'value' (USD)
                        # Controls both edge width and ranking order

# Ship type colors
SHIP_TYPE_COLORS = {
    'tanker': {'name': 'Tanker', 'color': '#8B4513'},       # Brown
    'bulk carrier': {'name': 'Bulk Carrier', 'color': '#FFD700'},  # Gold
    'cargo ship': {'name': 'Cargo Ship', 'color': '#4169E1'}  # Royal Blue
}

# Default color for ships with no type data
DEFAULT_SHIP_COLOR = 'gray'

In [5]:
# ==============================================================================
# LOAD DATA
# ==============================================================================
print("Loading network and simulation data...")

# Load HS codes mapping
with open('../data/hs_codes_mapping.json', 'r') as f:
    hs_codes_mapping = json.load(f)

print(f"✓ Loaded HS codes mapping: {len(hs_codes_mapping)} commodities")

# Load network
with open('../part_3_network_extraction/network_outputs/network_calibrated.gpickle', 'rb') as f:
    G_ch = pickle.load(f)

print(f"✓ Network loaded: {G_ch.number_of_nodes()} nodes, {G_ch.number_of_edges()} edges")

# Load ship locations (parquet)
_locs_df = pd.read_parquet(f'{SIMULATION_DIR}/ship_locations.parquet')
print(f"✓ Ship locations loaded: {len(_locs_df):,} records")
# Rename port_name -> port to match expected key in downstream cells
_locs_df = _locs_df.rename(columns={'port_name': 'port'})

# Optimise dtypes BEFORE groupby — critical for memory efficiency.
# Converting string columns with few unique values to Categorical reduces
# memory ~10-20x (only unique strings stored once; rows store integer codes).
print("  Optimising dtypes for memory efficiency...")
_locs_df['status']            = _locs_df['status'].astype('category')
_locs_df['node1']             = _locs_df['node1'].fillna('').astype('category')
_locs_df['node2']             = _locs_df['node2'].fillna('').astype('category')
_locs_df['port']              = _locs_df['port'].fillna('').astype('category')
_locs_df['progress_fraction'] = _locs_df['progress_fraction'].fillna(0.0).astype('float32')
_locs_df['ship_id']           = _locs_df['ship_id'].astype('int32')

# Index by timestep: {timestep: DataFrame_slice}
# Storing DataFrames (backed by typed NumPy arrays) instead of lists of Python
# dicts avoids ~40 GB of dict-object overhead for 66 M records.
print("  Indexing by timestep (this may take a moment)...")
ship_locations = {
    ts: grp.reset_index(drop=True)
    for ts, grp in _locs_df.groupby('timestep')
}
del _locs_df  # free the unsplit DataFrame
print(f"✓ Ship locations indexed: {len(ship_locations)} timesteps")

# Load ship data
ship_data_df = pd.read_csv(f'{SIMULATION_DIR}/compat/simulation_ship_data.csv')
print(f"✓ Ship data loaded: {len(ship_data_df)} ships")

# Extract HS codes present in data
hs_codes_in_data = []
for col in ship_data_df.columns:
    if col.startswith('cargo_hs') and col.endswith('_weight'):
        hs_code = int(col.replace('cargo_hs', '').replace('_weight', ''))
        hs_codes_in_data.append(hs_code)

print(f"✓ Found {len(hs_codes_in_data)} HS codes in ship data: {sorted(hs_codes_in_data)}")

# Create mappings for ship data
ship_weight_map = dict(zip(ship_data_df['ship_id'], ship_data_df['cargo_total_weight']))
ship_value_map = dict(zip(ship_data_df['ship_id'], ship_data_df['cargo_total_value']))
ship_type_map = dict(zip(ship_data_df['ship_id'], ship_data_df['ship_type']))

# Create ship color map based on ship type
ship_color_map = {}
for ship_id, ship_type in ship_type_map.items():
    if ship_type in SHIP_TYPE_COLORS:
        ship_color_map[ship_id] = SHIP_TYPE_COLORS[ship_type]['color']
    else:
        ship_color_map[ship_id] = DEFAULT_SHIP_COLOR

# Calculate cargo statistics for scaling (both weight and value)
min_weight = ship_data_df['cargo_total_weight'].min()
max_weight = ship_data_df['cargo_total_weight'].max()
mean_weight = ship_data_df['cargo_total_weight'].mean()

min_value = ship_data_df['cargo_total_value'].min()
max_value = ship_data_df['cargo_total_value'].max()
mean_value = ship_data_df['cargo_total_value'].mean()

print(f"\nCargo weight statistics:")
print(f"  Min: {min_weight:,.0f} tons")
print(f"  Max: {max_weight:,.0f} tons")
print(f"  Mean: {mean_weight:,.0f} tons")

print(f"\nCargo value statistics:")
print(f"  Min: ${min_value:,.0f}")
print(f"  Max: ${max_value:,.0f}")
print(f"  Mean: ${mean_value:,.0f}")

# Show ship type distribution
print(f"\nShip type distribution:")
for ship_type_key, info in SHIP_TYPE_COLORS.items():
    count = sum(1 for st in ship_type_map.values() if st == ship_type_key)
    percentage = 100 * count / len(ship_data_df) if len(ship_data_df) > 0 else 0
    print(f"  {info['name']}: {count} ships ({percentage:.1f}%) - Color: {info['color']}")

# Create HS code to weight/value mappings for each ship
ship_hs_weights = {}  # ship_id -> {hs_code: weight}
ship_hs_values = {}   # ship_id -> {hs_code: value}

for idx, row in ship_data_df.iterrows():
    ship_id = row['ship_id']
    ship_hs_weights[ship_id] = {}
    ship_hs_values[ship_id] = {}
    
    for hs_code in hs_codes_in_data:
        weight_col = f'cargo_hs{hs_code}_weight'
        value_col = f'cargo_hs{hs_code}_value'
        
        if weight_col in row and value_col in row:
            ship_hs_weights[ship_id][hs_code] = row[weight_col]
            ship_hs_values[ship_id][hs_code] = row[value_col]

# Load port occupancy data
port_occupancy_df = pd.read_csv(f'{SIMULATION_DIR}/compat/simulation_port_occupancy.csv')
print(f"\n✓ Port occupancy loaded: {len(port_occupancy_df)} records")

# Create port occupancy lookup: timestep -> {port_name: (num_ships, capacity)}
port_occupancy_by_timestep = {}
for _, row in port_occupancy_df.iterrows():
    timestep = int(row['timestep'])
    port_name = row['port_name']
    num_ships = int(row['num_ships'])
    capacity = int(row['capacity'])
    
    if timestep not in port_occupancy_by_timestep:
        port_occupancy_by_timestep[timestep] = {}
    port_occupancy_by_timestep[timestep][port_name] = (num_ships, capacity)

print(f"✓ Port occupancy indexed by {len(port_occupancy_by_timestep)} timesteps")

# Load world map for visualization
world = gpd.read_file('https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip')
print("✓ World map loaded")

print("\n" + "="*70)
print(f"EDGE METRIC: {EDGE_METRIC.upper()}")
print(f"  Edge widths will represent cumulative {EDGE_METRIC}")
print(f"  Rankings will be ordered by {EDGE_METRIC}")
print("="*70)

Loading network and simulation data...
✓ Loaded HS codes mapping: 97 commodities
✓ Network loaded: 8726 nodes, 14634 edges
✓ Ship locations loaded: 99,180,646 records
  Optimising dtypes for memory efficiency...
  Indexing by timestep (this may take a moment)...
✓ Ship locations indexed: 8759 timesteps
✓ Ship data loaded: 226302 ships
✓ Found 96 HS codes in ship data: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97]

Cargo weight statistics:
  Min: 8,969 tons
  Max: 227,007 tons
  Mean: 59,577 tons

Cargo value statistics:
  Min: $916,789
  Max: $69,163,186,050
  Mean: $85,764,956

Ship type distribution:
  Tanker: 115490 ships (51.0%) - Color: #8B4513
  Bulk Carri

In [6]:
# ==============================================================================
# CREATE ANIMATED VIDEO VISUALIZATION
# ==============================================================================
print("Creating animated video visualization...")
print("This may take several minutes...\n")

# Scaling parameters (defined early so they can be used in debug prints)
MIN_SHIP_SIZE = 3
MAX_SHIP_SIZE = 30
MIN_EDGE_WIDTH = 0.3
MAX_EDGE_WIDTH = 10.0
EDGE_NO_TRAFFIC_COLOR = '#CCCCCC'   # light gray for unused lanes
EDGE_NO_TRAFFIC_ALPHA = 0.07        # nearly invisible
EDGE_WITH_TRAFFIC_ALPHA = 1.0
PORT_NODE_SIZE = 8                  # small square port markers

# Convert node ID from string to appropriate type (int or str)
def convert_node_id(node):
    '''Convert node ID to appropriate type (int or str)'''
    node_str = str(node).strip()
    if node_str.startswith('port_') or node_str.startswith('choke_'):
        return node_str
    try:
        return int(node_str)
    except (ValueError, TypeError):
        return node_str

# Build canonical edge lookup
print("Building canonical edge lookup...")
canonical_edges = {}
for u, v in G_ch.edges():
    canonical_edges[(u, v)] = (u, v)
    canonical_edges[(v, u)] = (u, v)

print(f"Canonical edge lookup built: {len(canonical_edges)} entries ({G_ch.number_of_edges()} unique edges)\n")

# Create port name to node mapping
port_name_to_node = {}
for node in G_ch.nodes():
    if G_ch.nodes[node].get('source') == 'port':
        port_name = G_ch.nodes[node].get('portName')
        if port_name:
            port_name_to_node[port_name] = node

# Get network extent
lons = [G_ch.nodes[n]['lon'] for n in G_ch.nodes()]
lats = [G_ch.nodes[n]['lat'] for n in G_ch.nodes()]
lon_min, lon_max = min(lons) - 1, max(lons) + 1
lat_min, lat_max = min(lats) - 1, max(lats) + 1

# Convert ship_locations dict to sorted list of (timestep, DataFrame)
all_timesteps = sorted(ship_locations.items())   # list of (ts, DataFrame)

# Limit to N_FRAMES if specified
if N_FRAMES is not None and N_FRAMES < len(all_timesteps):
    timesteps = all_timesteps[:N_FRAMES]
    print(f"Limiting animation to first {N_FRAMES} frames (out of {len(all_timesteps)} total)")
else:
    timesteps = all_timesteps
    print(f"Creating animation with all {len(timesteps)} frames")

video_duration = len(timesteps) / FPS
print(f"Video duration: {video_duration:.1f} seconds at {FPS} fps")
first_day = timesteps[0][0]
last_day  = timesteps[-1][0]
print(f"Timestep range: Day {first_day} to Day {last_day}\n")

# Helper function for port colors
def get_port_color(num_ships, capacity):
    '''Return port color based on occupancy percentage'''
    if num_ships == 0:
        return 'green'
    occupancy_pct = num_ships / capacity if capacity > 0 else 0
    if occupancy_pct < 0.5:
        return 'green'
    elif occupancy_pct < 1.0:
        return 'orange'
    else:
        return 'red'

# ==============================================================================
# PRE-COMPUTE EDGE CARGO AND COMPLETED SHIPS
# ==============================================================================
print("Pre-computing edge cargo accumulation and completed ships tracking...")

edge_cargo_by_frame     = []
completed_ships_by_frame = []
completed_cargo_by_frame = []

cumulative_edge_cargo = {}
ship_edge_history     = {}
completed_ships       = set()
completed_hs_weights  = {}
completed_hs_values   = {}

for frame_idx, (ts, locations_data) in enumerate(tqdm(timesteps, desc="Computing data")):
    # locations_data is a DataFrame with columns:
    #   ship_id (int32), timestep, status (category), node1 (category),
    #   node2 (category), progress_fraction (float32), port (category)

    # DEBUG: Print header for first 3 frames
    if frame_idx < 3:
        print(f"\n{'='*60}")
        print(f"FRAME {frame_idx} (ts={ts}, {len(locations_data)} ships)")
        print(f"{'='*60}")

    # -- Track ships entering edges ------------------------------------------
    # Filter to active ships only, then iterate with itertuples (fast, low overhead)
    active_df = locations_data[locations_data['status'] == 'active']
    for loc in active_df.itertuples(index=False):
        ship_id_int = int(loc.ship_id)
        node1 = convert_node_id(loc.node1)
        node2 = convert_node_id(loc.node2)
        edge_key = canonical_edges.get((node1, node2))

        if edge_key is None:
            continue

        if ship_id_int not in ship_edge_history:
            ship_edge_history[ship_id_int] = set()

        if edge_key not in ship_edge_history[ship_id_int]:
            ship_edge_history[ship_id_int].add(edge_key)

            if edge_key not in cumulative_edge_cargo:
                cumulative_edge_cargo[edge_key] = {'weight': 0.0, 'value': 0.0}

            if frame_idx < 3:
                prev_weight = cumulative_edge_cargo[edge_key]['weight']
                ship_weight = ship_weight_map.get(ship_id_int, 0)
                print(f"  Ship {ship_id_int} → edge {edge_key} | "
                      f"weight: {prev_weight:,.0f} + {ship_weight:,.0f}")

            if ship_id_int in ship_weight_map:
                cumulative_edge_cargo[edge_key]['weight'] += ship_weight_map[ship_id_int]
            if ship_id_int in ship_value_map:
                cumulative_edge_cargo[edge_key]['value']  += ship_value_map[ship_id_int]

    # -- Track completed ships ------------------------------------------------
    if frame_idx > 0:
        _, prev_locations = timesteps[frame_idx - 1]
        # Build set of ship_ids active in the current frame for fast lookup
        current_ship_ids = set(locations_data['ship_id'].values)

        unloading_prev = prev_locations[prev_locations['status'] == 'unloading']
        for loc in unloading_prev.itertuples(index=False):
            sid = int(loc.ship_id)
            if sid not in current_ship_ids and sid not in completed_ships:
                completed_ships.add(sid)

                for hs_code, weight in ship_hs_weights.get(sid, {}).items():
                    completed_hs_weights[hs_code] = completed_hs_weights.get(hs_code, 0.0) + weight

                for hs_code, value in ship_hs_values.get(sid, {}).items():
                    completed_hs_values[hs_code] = completed_hs_values.get(hs_code, 0.0) + value

    # Store deep-copy snapshot for this frame
    frame_snapshot = {ek: {'weight': cd['weight'], 'value': cd['value']}
                      for ek, cd in cumulative_edge_cargo.items()}
    edge_cargo_by_frame.append(frame_snapshot)
    completed_ships_by_frame.append(set(completed_ships))
    completed_cargo_by_frame.append({
        'weight':     sum(completed_hs_weights.values()),
        'value':      sum(completed_hs_values.values()),
        'hs_weights': dict(completed_hs_weights),
        'hs_values':  dict(completed_hs_values),
    })

# Maximum edge cargo at end of video (used to scale widths throughout)
final_frame_edge_cargo = edge_cargo_by_frame[-1]
if final_frame_edge_cargo:
    max_edge_weight = max(e['weight'] for e in final_frame_edge_cargo.values())
    max_edge_value  = max(e['value']  for e in final_frame_edge_cargo.values())
else:
    max_edge_weight = max_edge_value = 0

print(f"\nData processing complete:")
print(f"  Unique edges with traffic: {len(cumulative_edge_cargo)}")
print(f"  Total ships completed:     {len(completed_ships)}")
print(f"  Max edge weight (final):   {max_edge_weight:,.0f} tons")
print(f"  Max edge value  (final):   ${max_edge_value:,.0f}")
print()

# Create figure
fig, ax = plt.subplots(figsize=(20, 16), dpi=100)
pos = nx.get_node_attributes(G_ch, 'pos')

def init():
    ax.clear()
    world.plot(ax=ax, color='lightgray', edgecolor='white', linewidth=0.5)
    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)

    for (u, v) in G_ch.edges():
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        dlon = x2 - x1
        if abs(dlon) > 180:
            if dlon > 0:
                frac = (-180 - x1) / (x2 - 360 - x1)
                y_b = y1 + frac * (y2 - y1)
                ax.plot([x1, -180], [y1, y_b], color=EDGE_NO_TRAFFIC_COLOR, linewidth=MIN_EDGE_WIDTH, alpha=EDGE_NO_TRAFFIC_ALPHA, linestyle='--', zorder=1)
                ax.plot([180, x2],  [y_b, y2], color=EDGE_NO_TRAFFIC_COLOR, linewidth=MIN_EDGE_WIDTH, alpha=EDGE_NO_TRAFFIC_ALPHA, linestyle='--', zorder=1)
            else:
                frac = (180 - x1) / (x2 + 360 - x1)
                y_b = y1 + frac * (y2 - y1)
                ax.plot([x1, 180],  [y1, y_b], color=EDGE_NO_TRAFFIC_COLOR, linewidth=MIN_EDGE_WIDTH, alpha=EDGE_NO_TRAFFIC_ALPHA, linestyle='--', zorder=1)
                ax.plot([-180, x2], [y_b, y2], color=EDGE_NO_TRAFFIC_COLOR, linewidth=MIN_EDGE_WIDTH, alpha=EDGE_NO_TRAFFIC_ALPHA, linestyle='--', zorder=1)
        else:
            ax.plot([x1, x2], [y1, y2], color=EDGE_NO_TRAFFIC_COLOR, linewidth=MIN_EDGE_WIDTH, alpha=EDGE_NO_TRAFFIC_ALPHA, zorder=1)

    port_nodes = [n for n in G_ch.nodes() if G_ch.nodes[n].get('source') == 'port']
    port_pos = {n: pos[n] for n in port_nodes}
    nx.draw_networkx_nodes(G_ch, port_pos, nodelist=port_nodes,
                           node_color='black', node_size=PORT_NODE_SIZE, alpha=1.0,
                           node_shape='s', ax=ax)

    ax.set_facecolor('#E8F4F8')
    ax.grid(True, linestyle='--', alpha=0.3)
    return []

pbar = tqdm(total=len(timesteps), desc="Rendering frames")

def animate(frame_idx):
    ax.clear()
    world.plot(ax=ax, color='lightgray', edgecolor='white', linewidth=0.5)
    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)

    ts, locations_data = timesteps[frame_idx]
    # Use the actual simulation day stored in the data (float, e.g. 1.042 = day 1 hour 1)
    day = float(locations_data['day'].iloc[0])

    frame_edge_cargo    = edge_cargo_by_frame[frame_idx]
    frame_completed_cargo = completed_cargo_by_frame[frame_idx]
    max_edge_metric     = max_edge_weight if EDGE_METRIC == 'weight' else max_edge_value

    # Draw edges
    for (u, v) in G_ch.edges():
        edge_data    = frame_edge_cargo.get((u, v), {'weight': 0, 'value': 0})
        metric_value = edge_data.get(EDGE_METRIC, 0)

        if metric_value > 0 and max_edge_metric > 0:
            color = 'black'
            width = MIN_EDGE_WIDTH + (MAX_EDGE_WIDTH - MIN_EDGE_WIDTH) * np.sqrt(metric_value / max_edge_metric)
            alpha = EDGE_WITH_TRAFFIC_ALPHA
        else:
            color = EDGE_NO_TRAFFIC_COLOR
            width = MIN_EDGE_WIDTH
            alpha = EDGE_NO_TRAFFIC_ALPHA

        x1, y1 = pos[u]
        x2, y2 = pos[v]
        dlon = x2 - x1
        if abs(dlon) > 180:
            if dlon > 0:
                frac = (-180 - x1) / (x2 - 360 - x1)
                y_b = y1 + frac * (y2 - y1)
                ax.plot([x1, -180], [y1, y_b], color=color, linewidth=width, alpha=alpha, linestyle='--', zorder=1)
                ax.plot([180, x2],  [y_b, y2], color=color, linewidth=width, alpha=alpha, linestyle='--', zorder=1)
            else:
                frac = (180 - x1) / (x2 + 360 - x1)
                y_b = y1 + frac * (y2 - y1)
                ax.plot([x1, 180],  [y1, y_b], color=color, linewidth=width, alpha=alpha, linestyle='--', zorder=1)
                ax.plot([-180, x2], [y_b, y2], color=color, linewidth=width, alpha=alpha, linestyle='--', zorder=1)
        else:
            ax.plot([x1, x2], [y1, y2], color=color, linewidth=width, alpha=alpha, zorder=1)

    # Draw ports coloured by occupancy
    port_nodes = [n for n in G_ch.nodes() if G_ch.nodes[n].get('source') == 'port']
    port_pos   = {n: pos[n] for n in port_nodes}
    ports_by_color = {'green': [], 'orange': [], 'red': []}

    for node in port_nodes:
        port_name = G_ch.nodes[node].get('portName')
        num_ships, capacity = 0, 1
        if frame_idx in port_occupancy_by_timestep and port_name in port_occupancy_by_timestep.get(frame_idx, {}):
            num_ships, capacity = port_occupancy_by_timestep[frame_idx][port_name]
        ports_by_color[get_port_color(num_ships, capacity)].append(node)

    for color, nodes in ports_by_color.items():
        if nodes:
            nx.draw_networkx_nodes(G_ch, {n: port_pos[n] for n in nodes}, nodelist=nodes,
                                   node_color=color, node_size=PORT_NODE_SIZE, alpha=1.0,
                                   node_shape='s', ax=ax)

    # Draw ships — locations_data is a DataFrame; use itertuples for fast row access
    ships_by_type = {k: {'positions': [], 'sizes': []} for k in SHIP_TYPE_COLORS}

    for loc in locations_data.itertuples(index=False):
        ship_id_int = int(loc.ship_id)
        lon = lat = None

        status = str(loc.status)
        if status == 'active':
            node1 = convert_node_id(loc.node1)
            node2 = convert_node_id(loc.node2)
            pf    = float(loc.progress_fraction)
            try:
                lon1, lat1 = G_ch.nodes[node1]['lon'], G_ch.nodes[node1]['lat']
                lon2, lat2 = G_ch.nodes[node2]['lon'], G_ch.nodes[node2]['lat']
                dlon = lon2 - lon1
                if abs(dlon) > 180:
                    # Short-arc interpolation — ship teleports at antimeridian
                    lon2_uw = lon2 - 360 if dlon > 0 else lon2 + 360
                    lon = ((lon1 + pf * (lon2_uw - lon1) + 180) % 360) - 180
                else:
                    lon = lon1 + pf * (lon2 - lon1)
                lat = lat1 + pf * (lat2 - lat1)
            except KeyError:
                continue

        elif status in ('loading', 'unloading'):
            port_name = str(loc.port) if loc.port else None
            if port_name and port_name in port_name_to_node:
                pn = port_name_to_node[port_name]
                try:
                    lon = G_ch.nodes[pn]['lon']
                    lat = G_ch.nodes[pn]['lat']
                except KeyError:
                    continue

        if lon is None or lat is None:
            continue

        ship_type = ship_type_map.get(ship_id_int, 'cargo ship')
        weight    = ship_weight_map.get(ship_id_int, 0)
        size = (MIN_SHIP_SIZE + (MAX_SHIP_SIZE - MIN_SHIP_SIZE) *
                ((weight - min_weight) / (max_weight - min_weight))
                if max_weight > min_weight else MIN_SHIP_SIZE)

        if ship_type in ships_by_type:
            ships_by_type[ship_type]['positions'].append((lon, lat))
            ships_by_type[ship_type]['sizes'].append(size)

    markers = {'cargo ship': 'o', 'tanker': 's', 'bulk carrier': '^'}
    for ship_type, data in ships_by_type.items():
        if data['positions']:
            lons_, lats_ = zip(*data['positions'])
            ax.scatter(lons_, lats_, c=SHIP_TYPE_COLORS[ship_type]['color'],
                       s=data['sizes'], alpha=0.9,
                       edgecolors='black', linewidths=0.5,
                       marker=markers.get(ship_type, 'o'), zorder=10)

    ax.set_facecolor('#E8F4F8')
    ax.grid(True, linestyle='--', alpha=0.3)

    # Title: show actual simulation day (float) so fractional days are visible
    progress_pct = (frame_idx + 1) / len(timesteps) * 100
    bar_width = 40
    filled  = int(bar_width * (frame_idx + 1) / len(timesteps))
    bar     = '█' * filled + '░' * (bar_width - filled)
    ax.set_title(f'Day {day:.2f}\n[{bar}] {progress_pct:.1f}%',
                 fontsize=16, fontweight='bold', pad=20)

    # Completed shipments text box (top-left) — fully opaque background
    total_w = frame_completed_cargo['weight']
    total_v = frame_completed_cargo['value']
    chs_w   = frame_completed_cargo['hs_weights']
    chs_v   = frame_completed_cargo['hs_values']

    if EDGE_METRIC == 'weight':
        sorted_hs = sorted(chs_w.items(), key=lambda x: x[1], reverse=True)[:5]
    else:
        sorted_hs = sorted(chs_v.items(), key=lambda x: x[1], reverse=True)[:5]

    lines = ['COMPLETED SHIPMENTS',
             f'Weight: {total_w:,.0f} tons',
             f'Value: ${total_v:,.0f}', '',
             f'Top 5 by {EDGE_METRIC}:']
    for hs_code, _ in sorted_hs:
        hs_str  = str(hs_code).zfill(2)
        hs_name = hs_codes_mapping.get(hs_str, {}).get('name', f'HS{hs_str}')
        if len(hs_name) > 25:
            hs_name = hs_name[:22] + '...'
        val_str = (f'{chs_w.get(hs_code, 0):,.0f}t' if EDGE_METRIC == 'weight'
                   else f'${chs_v.get(hs_code, 0):,.0f}')
        lines.append(f'{hs_name}: {val_str}')

    # Cumulative stats — bottom-left
    ax.text(0.02, 0.02, '\n'.join(lines),
            transform=ax.transAxes, fontsize=10, verticalalignment='bottom',
            bbox=dict(boxstyle='round', facecolor='white', alpha=1.0, edgecolor='black'),
            zorder=20)

    # Ship type legend — bottom-right
    ax.text(0.98, 0.02, 'SHIP TYPES\n○ Cargo Ship\n■ Tanker\n▲ Bulk Carrier',
            transform=ax.transAxes, fontsize=10, verticalalignment='bottom',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=1.0, edgecolor='black'),
            zorder=20)

    pbar.update(1)
    return []

# Create animation
print(f"Building animation object ({FPS} fps)...")
anim = animation.FuncAnimation(fig, animate, init_func=init,
                               frames=len(timesteps),
                               interval=1000 // FPS, blit=True)

# Save animation
print(f"\nSaving animation ({len(timesteps)} frames at {FPS} fps)...")

saved = False

try:
    print("Attempting to save as GIF with Pillow...")
    from matplotlib.animation import PillowWriter
    writer = PillowWriter(fps=FPS)
    output_file = f'visualization_outputs/{SCENARIO}.gif'
    anim.save(output_file, writer=writer, dpi=80)
    pbar.close()
    print(f"✓ Animation saved as GIF: {output_file}")
    saved = True
except Exception as e:
    pbar.close()
    print(f"✗ Pillow GIF failed: {str(e)[:200]}")


# Always attempt MP4 as well (higher quality, smaller file)
try:
    print("\nAttempting to save as MP4 with ffmpeg...")
    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=FPS, metadata=dict(artist='Global Shipping Simulation'), bitrate=3000)
    output_file = f'visualization_outputs/{SCENARIO}.mp4'
    anim.save(output_file, writer=writer, dpi=100)
    print(f"✓ Video saved: {output_file}")
except Exception as e:
    print(f"✗ ffmpeg not available: {str(e)[:200]}")
    print("  Install with: brew install ffmpeg")

plt.close(fig)
print("\nDone!")

Creating animated video visualization...
This may take several minutes...

Building canonical edge lookup...
Canonical edge lookup built: 29268 entries (14634 unique edges)

Limiting animation to first 480 frames (out of 8759 total)
Video duration: 24.0 seconds at 20 fps
Timestep range: Day 1 to Day 480

Pre-computing edge cargo accumulation and completed ships tracking...


Computing data:  14%|█▍        | 66/480 [00:00<00:00, 648.46it/s]


FRAME 0 (ts=1, 28 ships)

FRAME 1 (ts=2, 49 ships)

FRAME 2 (ts=3, 66 ships)


Computing data: 100%|██████████| 480/480 [00:04<00:00, 119.06it/s]



Data processing complete:
  Unique edges with traffic: 7966
  Total ships completed:     4145
  Max edge weight (final):   56,507,057 tons
  Max edge value  (final):   $155,468,208,517



Rendering frames:   0%|          | 0/480 [00:00<?, ?it/s]

Building animation object (20 fps)...

Saving animation (480 frames at 20 fps)...
Attempting to save as GIF with Pillow...


Rendering frames: 100%|██████████| 480/480 [36:49<00:00,  4.60s/it]

✓ Animation saved as GIF: visualization_outputs/scenario_suez_evergiven.gif

Attempting to save as MP4 with ffmpeg...
✗ ffmpeg not available: Requested MovieWriter (ffmpeg) not available
  Install with: brew install ffmpeg

Done!
